### Data Loading and Preprocessing

In this step, we load the IMDB dataset for sentiment analysis  from the Kaggle CSV file. The dataset consists of movie reviews labeled as positive or negative.

Since the dataset contains raw text, we perform preprocessing steps before training:
- Convert all text to lowercase
- Remove HTML tags
- Remove punctuation and special characters
- Tokenize the text into sequences of integers
- Pad or truncate seqeunces to a fixed length of 256 tokens

The sentiment labels are encoded as binary values (0 = negative, 1 = positive). These steps prepare the data for input into the neural network.

In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_features = 10000
max_len = 256

# Load Kaggle dataset
df = pd.read_csv('IMDB Dataset.csv', engine='python',on_bad_lines='skip')

# Clean text
texts = df['review'].str.lower()
texts = texts.apply(lambda x: re.sub(r'<.*?>', '', x))

# Labels
labels = df['sentiment'].map({'positive': 1, 'negative': 0}).values

# Tokenize
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(texts)

X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=max_len)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, labels, test_size=0.2, random_state=42
)

In [ ]:
print(X_train.shape, y_train.shape)

### LSTM Model Architecture

We build a neural network using an embedding layer followed by an LSTM.

The embedding layer converts word indices into dense vector representations. The LSTM processes the sequence of words and captures contextual relationships. A dropout layer is used to reduce overfitting, and a final dense layer with sigmoid activation outputs a probability for binary classification.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(max_features, 128),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

### Model Compilation

We compile the model using the Adam optimizer and binary cross-entropy loss.

Binary cross-entropy is appropriate because this is a binary classification problem (positive vs negative sentiment). Accuracy is used as the evaluation metric.

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Model Training

The model is trained on the training data for a small number of epochs.

We use a validation split to monitor performance on unseen data during training. This helps evaluate how well the model generalizes.

history = model.fit(
    X_train,
    y_train,
    epochs=2,
    batch_size=64,
    validation_split=0.2
)

### Model Evaluation

After training, we evaluate the model on the test dataset.

The test accuracy provides an estimate of how well the model performs on unseen data.

loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

### Summary

We implemented an LSTM-based model for sentiment classification on the IMDB dataset. The model achieved approximately 84-87% test accuracy. Further improvements can be made through hyperparameter tuning and more advanced processing techniques.